# 07 | Modelos avanzados y ensambles - Sprint 4

## Objetivo del notebook

Este notebook explica y ejecuta los modelos avanzados pedidos en Sprint 4:

- XGBoost
- LightGBM
- Bagging
- Voting
- Stacking

La idea es probar si modelos más complejos capturan relaciones no lineales que los baselines no capturan.


In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


Project root: /Users/alexandralozano/dp261-g1-final 2


## 1. Por qué usar XGBoost y LightGBM

Ambos son modelos de gradient boosting, muy usados en datos tabulares porque:

- Capturan no linealidades.
- Manejan interacciones entre variables sin crearlas manualmente todas.
- Tienen regularización y muestreo para controlar overfitting.
- Permiten `scale_pos_weight` para compensar desbalance.

En este proyecto reciben variables categóricas con `OrdinalEncoder`, no OHE masivo, para reducir dimensionalidad y acelerar entrenamiento.


## 2. Por qué usar ensambles

| Ensamble | Idea | Cuándo ayuda |
|---|---|---|
| Bagging | Entrena varios árboles en subconjuntos aleatorios | Reduce varianza de árboles individuales. |
| Voting | Promedia probabilidades de modelos distintos | Ayuda si modelos cometen errores diferentes. |
| Stacking | Usa modelos base y un meta-modelo | Puede combinar fortalezas, pero aumenta complejidad. |

Más complejidad no garantiza mejor resultado. Por eso todos compiten contra la misma función de negocio.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.config import RAW_DATA_PATH, TARGET, RANDOM_STATE
from src.preprocessing import split_X_y
from src.models import build_advanced_models, build_ensembles
from src.evaluation import evaluate_model

df = pd.read_csv(RAW_DATA_PATH).sample(3000, random_state=RANDOM_STATE)
train, valid = train_test_split(df, test_size=0.30, stratify=df[TARGET], random_state=RANDOM_STATE)
X_train, y_train = split_X_y(train)
X_valid, y_valid = split_X_y(valid)
advanced = build_advanced_models(X_train, y_train)
ensembles = build_ensembles(X_train, y_train)
list(advanced.keys()) + list(ensembles.keys())


['XGBoost_baseline',
 'LightGBM_baseline',
 'Bagging_DecisionTree',
 'Voting_soft_LR_RF_LGBM',
 'Stacking_RF_LGBM_LR']

In [3]:
rows = []
for name, pipe in {**advanced, **ensembles}.items():
    pipe.fit(X_train, y_train)
    metrics = evaluate_model(pipe, X_valid, y_valid)
    metrics.pop('classification_report', None)
    rows.append({'model': name, **metrics})

advanced_results = pd.DataFrame(rows).sort_values('business_value', ascending=False)
advanced_results[['model','business_value','recall','precision','f2','roc_auc','tp','fp','fn','tn']]


/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/v

,model,business_value,recall,precision,f2,roc_auc,tp,fp,fn,tn
1,LightGBM_baseline,163300.0,0.333333,0.350515,0.336634,0.699365,34,63,68,735
2,Bagging_DecisionTree,154800.0,0.352941,0.315789,0.344828,0.702301,36,78,66,720
3,Voting_soft_LR_RF_LGBM,144800.0,0.431373,0.265060,0.383275,0.708966,44,122,58,676
4,Stacking_RF_LGBM_LR,108300.0,0.705882,0.206304,0.475561,0.720797,72,277,30,521
0,XGBoost_baseline,76800.0,0.382353,0.213115,0.329949,0.707823,39,144,63,654


## 3. Cómo decidir si un ensamble vale la pena

Un ensamble se justifica si:

1. Mejora el valor económico.
2. Reduce falsos negativos sin disparar falsos positivos.
3. Su complejidad se puede explicar al stakeholder.
4. La API puede cargarlo y predecir sin latencia excesiva.

Si un LightGBM simple gana o queda muy cerca, puede ser preferible por simplicidad operativa.
